# Sense-Think-Act Assignment

The goal of this assignment is to implement a simple robot control algorithm that can detect light intensity and spin the robot in response to high intensity light.

## 📚 Schedule of Notebooks

Try to complete the following notebooks in the order listed below. Each notebook has an estimated duration.

1. 🚀 [Getting Started](../week-3/01-getting-started.ipynb) - **60 min**
2. 🔧 [Rclpy Snippets](../week-3/02-rclpy-snippets.ipynb) - **60 min**
3. 📋 [Assignment](03-sta-assignment.ipynb) - **120 min**

## Demonstration

The following video demonstrates the expected robot's behaviour:

<iframe width="560" height="315" src="https://www.youtube.com/embed/kxwQ-6f3SEI?si=eoZRijScd-9a150t" title="STA Assignment Demo" frameborder="0" allow="accelerometer; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" allowfullscreen></iframe>

Can't see the video? [Link to the video](https://www.youtube.com/watch?v=kxwQ-6f3SEI)

## Method

To complete the assignment, you must first break down the project into smaller, manageable components.

1. Reading the ambient light sensor
1. Identifying the threshold LUX value for the robot spin
1. Controlling the motors (actuators) on RVR
1. Spinning the RVR
1. Logic to connect sensor input to the actuation output
1. Putting it all together within a 'Sense, Think, Act' loop

#### Light Sensor

The light sensor is located on the front of the RVR. It is a digital sensor that returns a number. The higher the value, the brighter the light.

#### Threshold Value

The threshold value is the value at which the robot will spin. You will need to experiment with different values to find the best one.

#### Motor Control

The rover responds to twist commands. The twist command is a combination of linear and angular velocity. The linear velocity is the speed at which the robot moves forward or backward. The angular velocity is the speed at which the robot spins.

The `cmd_vel` topic is a standard topic used to send twist commands to a robot. The message type is `geometry_msgs/Twist`.

---
## Step 1 — ROS 2 Network Configuration

Set the environment variables so this notebook can communicate with the RVR over the shared Wi-Fi network.

> ⚠️ **Change `ROBOT_HOSTNAME` and `CONSOLE_HOSTNAME` to match your group's robot and laptop IP addresses before running any cells.**

In [ ]:
"""
Step 1: ROS 2 networking setup for shared Wi-Fi with discovery server.
Must be run FIRST, before importing rclpy or creating any nodes.
"""

import os

#####################################################################
# !!!! DON'T FORGET TO CHANGE THESE TO MATCH YOUR ROBOT !!!! #
#####################################################################

ROBOT_HOSTNAME   = "192.168.1.101"   # robot (discovery server) IP address
CONSOLE_HOSTNAME = "192.168.1.201"   # this laptop/container IP address

DISCOVERY_SERVER_PORT = 11811        # FastDDS discovery server port
DISCOVERY_SERVER_HOST = ROBOT_HOSTNAME
ROS_DOMAIN_ID         = "23"         # must match the robot's domain ID

# Apply networking environment variables
os.environ["ROS_DOMAIN_ID"]                = ROS_DOMAIN_ID
os.environ["ROS_HOSTNAME"]                 = CONSOLE_HOSTNAME
os.environ["RMW_IMPLEMENTATION"]           = "rmw_fastrtps_cpp"
os.environ["ROS_AUTOMATIC_DISCOVERY_RANGE"] = "SUBNET"
os.environ["ROS_DISCOVERY_SERVER"]         = f"{DISCOVERY_SERVER_HOST}:{DISCOVERY_SERVER_PORT}"
os.environ["ROS_SUPER_CLIENT"]             = "True"

print(f"ROS_DOMAIN_ID      : {os.environ['ROS_DOMAIN_ID']}")
print(f"ROS_HOSTNAME       : {os.environ['ROS_HOSTNAME']}")
print(f"ROS_DISCOVERY_SERVER: {os.environ['ROS_DISCOVERY_SERVER']}")
print("Environment configured ✓")

---
## Step 2 — Initialise rclpy

Initialise the ROS 2 Python client library. This must be done once per kernel session, before creating any nodes.

In [ ]:
"""
Step 2: Initialise rclpy (ROS 2 Python client library).
Safe to call multiple times — the guard prevents double-initialisation.
"""

import rclpy

if not rclpy.ok():
    rclpy.init()
    print("rclpy initialised ✓")
else:
    print("rclpy already running ✓")

---
## Step 3 — Read the Ambient Light Sensor

Before writing the full algorithm, let's verify that we can receive data from the light sensor topic.

The RVR publishes ambient light (illuminance) values on the topic:
```
/light_sensor_broadcaster/illuminance
```
The message type is `sensor_msgs/Illuminance`. The key field is `illuminance`, which contains the value in **Lux**.

> ⚠️ This cell runs forever. Use **Kernel → Restart** to stop it, then re-run Steps 1 and 2 before continuing.

In [ ]:
"""
Step 3: Subscribe to the light sensor topic and print the illuminance value.
Use this cell to discover what LUX value corresponds to "bright light" in your lab.
"""

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Illuminance


class LightSensorReaderNode(Node):
    """Subscribes to the illuminance topic and prints each reading."""

    def __init__(self):
        super().__init__('light_sensor_reader_node')
        self.light_sub = self.create_subscription(
            Illuminance,
            '/light_sensor_broadcaster/illuminance',
            self.light_callback,
            10
        )
        self.get_logger().info('LightSensorReaderNode started — listening for illuminance...')

    def light_callback(self, msg: Illuminance):
        """Called every time a new illuminance message arrives."""
        lux = msg.illuminance
        print(f'Illuminance: {lux:.2f} lux')


def main():
    print('Starting LightSensorReaderNode...')
    try:
        node = LightSensorReaderNode()
        rclpy.spin(node)          # blocks; prints readings as they arrive
    except KeyboardInterrupt:
        print('Interrupted')
    finally:
        print('Node shutdown.')


print('LightSensorReaderNode declared — call main() to run it')

In [ ]:
# Run this cell to start listening to the light sensor
# Watch the printed lux values. Shine a torch/phone light at the RVR sensor
# and note the values — use those to calibrate LIGHT_THRESHOLD below.
main()

---
## Step 4 — Set the Threshold LUX Value

Based on the values you observed in Step 3, choose a threshold. The robot will spin whenever the measured lux **exceeds** this value.

| Environment | Typical LUX range |
|---|---|
| Normal indoor room | 100 – 500 lux |
| Phone torch close-up | 2 000 – 10 000 lux |

Start with `LIGHT_THRESHOLD = 500` and adjust after testing.

In [ ]:
"""
Step 4: Define the LUX threshold.
Adjust this value based on the readings you observed in Step 3.
"""

# ------------------------------------------------------------------ #
# !! CHANGE THIS VALUE based on your observed sensor readings !!      #
# ------------------------------------------------------------------ #
LIGHT_THRESHOLD = 500.0   # lux — robot spins when illuminance > this value

print(f'Light threshold set to {LIGHT_THRESHOLD} lux')

---
## Step 5 — Motor Control: Test Spinning the RVR

Before combining everything, verify that the motor control works. The RVR accepts `geometry_msgs/TwistStamped` messages on `/cmd_vel`.

- `twist.linear.x` → forward/backward speed (m/s)
- `twist.angular.z` → spin speed (rad/s) — positive = counter-clockwise

> ⚠️ This cell runs forever. Use **Kernel → Restart** to stop it, then re-run Steps 1 and 2 before continuing.

In [ ]:
"""
Step 5: Publish a constant spin command to /cmd_vel to confirm motor control works.
The robot should spin in place while this cell runs.
"""

import rclpy
from rclpy.node import Node
from geometry_msgs.msg import TwistStamped
from time import sleep


class SpinTestNode(Node):
    """Publishes a fixed spin command at 2 Hz to test motor control."""

    ANGULAR_SPEED = 1.0   # rad/s — change sign to reverse spin direction
    PUBLISH_RATE  = 2.0   # Hz

    def __init__(self):
        super().__init__('spin_test_node')
        self.cmd_vel_pub = self.create_publisher(TwistStamped, '/cmd_vel', 10)
        self.get_logger().info('SpinTestNode started')

    def run(self):
        """Publish spin command in a manual loop."""
        print(f'Spinning at {self.ANGULAR_SPEED} rad/s — press Stop / restart kernel to halt')
        while rclpy.ok():
            msg = TwistStamped()
            msg.twist.linear.x  = 0.0
            msg.twist.angular.z = self.ANGULAR_SPEED
            self.cmd_vel_pub.publish(msg)
            sleep(1.0 / self.PUBLISH_RATE)

    def stop(self):
        """Publish a zero-velocity command to halt the robot."""
        stop_msg = TwistStamped()
        stop_msg.twist.linear.x  = 0.0
        stop_msg.twist.angular.z = 0.0
        self.cmd_vel_pub.publish(stop_msg)
        print('Stop command sent ✓')


def main():
    print('Starting SpinTestNode...')
    node = SpinTestNode()
    try:
        node.run()
    except KeyboardInterrupt:
        print('Interrupted')
    finally:
        node.stop()
        print('Node shutdown.')


print('SpinTestNode declared — call main() to run it')

In [ ]:
# Run this cell to test spinning — the robot should spin in place
main()

---
## Step 6 — Putting it all together: the Sense-Think-Act Node

Now we combine everything into a single ROS node with a clean **Sense → Think → Act** loop:

| Phase | What happens |
|---|---|
| **Sense** | Light sensor callback stores the latest lux reading |
| **Think** | Timer callback compares lux to `LIGHT_THRESHOLD` |
| **Act** | Publisher sends a spin or stop command accordingly |

The timer fires at 10 Hz, giving a responsive control loop without blocking the subscriber.

In [ ]:
"""
Step 6 — Sense-Think-Act Node

SENSE  : Subscribe to /light_sensor_broadcaster/illuminance
THINK  : Compare latest lux reading to LIGHT_THRESHOLD
ACT    : Publish spin or stop command to /cmd_vel
"""

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Illuminance
from geometry_msgs.msg import TwistStamped


class SenseThinkActNode(Node):
    """
    Implements a Sense-Think-Act control loop for the Sphero RVR.

    The robot spins whenever the ambient light sensor reads above
    LIGHT_THRESHOLD lux, and stops otherwise.
    """

    # -------------------------------------------------------------- #
    # Configuration — adjust these values as needed                   #
    # -------------------------------------------------------------- #
    LIGHT_THRESHOLD = 500.0   # lux — spin when illuminance exceeds this
    ANGULAR_SPEED   = 1.5     # rad/s — spin speed (positive = CCW)
    LOOP_RATE_HZ    = 10.0    # Hz — how often the think/act timer fires

    def __init__(self):
        super().__init__('sense_think_act_node')

        # ---- SENSE: light sensor subscriber ---- #
        self.light_sub = self.create_subscription(
            Illuminance,
            '/light_sensor_broadcaster/illuminance',
            self._sense_cb,       # called every time a new reading arrives
            10
        )

        # ---- ACT: velocity command publisher ---- #
        self.cmd_vel_pub = self.create_publisher(TwistStamped, '/cmd_vel', 10)

        # ---- THINK: timer drives the control loop ---- #
        self.timer = self.create_timer(
            1.0 / self.LOOP_RATE_HZ,
            self._think_and_act_cb
        )

        # State: latest illuminance reading (starts at 0 — robot is still)
        self.current_lux: float = 0.0

        self.get_logger().info(
            f'SenseThinkActNode started | threshold={self.LIGHT_THRESHOLD} lux | '
            f'spin speed={self.ANGULAR_SPEED} rad/s | loop={self.LOOP_RATE_HZ} Hz'
        )

    # ============================================================== #
    # SENSE                                                           #
    # ============================================================== #
    def _sense_cb(self, msg: Illuminance):
        """
        Callback: receives the latest illuminance reading and stores it.
        This is the SENSE phase — we simply store what the sensor reports.
        """
        self.current_lux = msg.illuminance
        self.get_logger().debug(f'Sensed: {self.current_lux:.1f} lux')

    # ============================================================== #
    # THINK + ACT                                                     #
    # ============================================================== #
    def _think_and_act_cb(self):
        """
        Timer callback: runs at LOOP_RATE_HZ.

        THINK — decide what to do based on the current sensor reading.
        ACT   — publish the appropriate velocity command.
        """
        cmd = TwistStamped()

        # ---- THINK ---- #
        light_is_bright = self.current_lux > self.LIGHT_THRESHOLD

        if light_is_bright:
            # ---- ACT: spin ---- #
            cmd.twist.linear.x  = 0.0
            cmd.twist.angular.z = self.ANGULAR_SPEED
            self.get_logger().info(
                f'BRIGHT ({self.current_lux:.1f} lux > {self.LIGHT_THRESHOLD}) → SPINNING'
            )
        else:
            # ---- ACT: stop ---- #
            cmd.twist.linear.x  = 0.0
            cmd.twist.angular.z = 0.0
            self.get_logger().info(
                f'DIM   ({self.current_lux:.1f} lux ≤ {self.LIGHT_THRESHOLD}) → STOPPED'
            )

        self.cmd_vel_pub.publish(cmd)

    # ============================================================== #
    # CLEANUP                                                         #
    # ============================================================== #
    def stop_robot(self):
        """Publish a zero-velocity command to safely halt the robot."""
        stop_cmd = TwistStamped()
        stop_cmd.twist.linear.x  = 0.0
        stop_cmd.twist.angular.z = 0.0
        self.cmd_vel_pub.publish(stop_cmd)
        self.get_logger().info('Safety stop sent ✓')


# ------------------------------------------------------------------ #
# Main                                                                #
# ------------------------------------------------------------------ #
def main():
    print('Starting SenseThinkActNode...')
    print(f'  Threshold : {SenseThinkActNode.LIGHT_THRESHOLD} lux')
    print(f'  Spin speed: {SenseThinkActNode.ANGULAR_SPEED} rad/s')
    print(f'  Loop rate : {SenseThinkActNode.LOOP_RATE_HZ} Hz')
    print()

    node = SenseThinkActNode()
    try:
        # rclpy.spin() drives both the subscriber callback and the timer callback
        rclpy.spin(node)
    except KeyboardInterrupt:
        print('Interrupted')
    finally:
        node.stop_robot()   # always stop the robot on exit
        print('Node shutdown.')


print('SenseThinkActNode declared ✓')

### Run the Sense-Think-Act Loop

Execute the cell below to start the full algorithm.

- **Shine a light** at the RVR's front sensor → the robot should start spinning.
- **Remove the light** → the robot should stop.
- If the robot always spins or never spins, adjust `LIGHT_THRESHOLD` in the class definition above.

> ⚠️ Use **Kernel → Restart** to stop the node. After restarting, re-run Steps 1 and 2 before using any ROS cells.

In [ ]:
# Run the full Sense-Think-Act algorithm
main()